In [ ]:
# Замена поиска по ключевым словам на векторный поиск в нашем конвейере RAG.

In [4]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [5]:
from rag_helper import RAGBase

assistant = RAGBase(
  index = index,
  llm_client = openai_client,
)

In [7]:
# В данном пример используется поиск по ключевым словам
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

'Yes, you can still join. If you want to receive a certificate, make sure to submit your project while submissions are still being accepted.'

In [8]:
# Однако его необходимо заменить векторным поиском
# Он расширяет наш основной класс
class RAGVector(RAGBase):
  def __init__(self, embedder, **kwargs):
    super().__init__(**kwargs)
    self.embedder = embedder

  def search(self, query, num_results = 5):
    query_vector = self.embedder.encode(query)
    filter_dict = {"course": self.course}

    return self.index.search(
      query_vector,
      num_results = num_results,
      filter_dict = filter_dict
    )

In [12]:
texts = []

for doc in documents:
  text = doc["question"] + " " + doc["answer"]
  texts.append(text)

In [13]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
  batch = texts[i:i + batch_size]
  batch_vectors = model.encode(batch)
  vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/28 [00:00<?, ?it/s]

1400

In [14]:
import numpy as np

X = np.array(vectors)

In [15]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [16]:
vector_assistant = RAGVector(
  embedder = model,
  index = vindex,
  llm_client = openai_client,
)

In [17]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes — you can still join. If you want a certificate, make sure to submit your project while submissions are still being accepted.'

In [ ]:
# Однако у данной методике есть 3 проблемы:
# - Индекс перестраивается при каждом запуске.
# - Оно хранит все данные в памяти.
# - Поиск осуществляется методом перебора - сравниваем вектор
# запроса с каждым отдельным документом

# При векторном поиске индексирование запускает 
# нейронную сеть по каждому документу, поэтому 
# на нашем наборе данных это занимает минуту.